In [2]:
import pandas as pd
import duckdb
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.utils.data_loader import load_datasets

dfs = load_datasets(["vendas_2023_2024.csv"], layer="raw")
df_sales = dfs["vendas_2023_2024.csv"]

df_sales["sale_date"] = pd.to_datetime(df_sales["sale_date"], format="mixed", dayfirst=True).dt.date

In [17]:
pd.options.display.float_format = 'R$ {:,.2f}'.format

query_least_profit_days = """
WITH RECURSIVE calendar AS (
    SELECT
        (SELECT MIN(sale_date)::DATE FROM df_sales) as dim_date,
        (SELECT MAX(sale_date)::DATE FROM df_sales) as end_date

    UNION ALL

    SELECT
        (dim_date + INTERVAL '1 day')::DATE AS dim_date,
        end_date
    FROM calendar
    WHERE dim_date < end_date
),
calendar_with_dow AS (
    SELECT
        dim_date,
        CASE DAYOFWEEK(dim_date)
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-Feira'
            WHEN 2 THEN 'Terça-Feira'
            WHEN 3 THEN 'Quarta-Feira'
            WHEN 4 THEN 'Quinta-Feira'
            WHEN 5 THEN 'Sexta-Feira'
            WHEN 6 THEN 'Sábado'
        END AS day_name_pt
    FROM calendar
),
daily_sales AS (
    SELECT
        sale_date,
        SUM(total) AS daily_revenue,
    FROM df_sales
    GROUP BY sale_date
),
calendar_joined AS (
    SELECT
        c.dim_date,
        c.day_name_pt,
        COALESCE(ds.daily_revenue, 0) AS final_daily_revenue
    FROM calendar_with_dow c
    LEFT JOIN daily_sales ds
        ON c.dim_date = ds.sale_date
)
SELECT 
    day_name_pt,
    AVG(final_daily_revenue) AS true_average_sales
FROM calendar_joined
GROUP BY day_name_pt
ORDER BY true_average_sales ASC;
"""

df_calendar_sales = duckdb.query(query_least_profit_days).df()

worst_day = df_calendar_sales.iloc[0]

print("=" * 51)
print(f"O dia da semana com a PIOR média histórica de vendas é o: {worst_day['day_name_pt'].upper()}")
print(f"Média diária real (considerando dias zerados): R$ {worst_day['true_average_sales']:,.2f}")
print("=" * 51, "\n")

df_presentation = df_calendar_sales.rename(columns={
    'day_name_pt': 'Dia da Semana',
    'true_average_sales': 'Média Histórica de Vendas (R$)'
})

print("Ranking Completo:")
print(df_presentation.to_string(index=False))

O dia da semana com a PIOR média histórica de vendas é o: DOMINGO
Média diária real (considerando dias zerados): R$ 3,229,614.16

Ranking Completo:
Dia da Semana  Média Histórica de Vendas (R$)
      Domingo                 R$ 3,229,614.16
Segunda-Feira                 R$ 3,484,500.47
  Terça-Feira                 R$ 3,488,871.99
 Quarta-Feira                 R$ 3,534,007.21
 Quinta-Feira                 R$ 3,713,299.94
       Sábado                 R$ 3,774,290.79
  Sexta-Feira                 R$ 3,776,151.25
